# 003_pdf.ipynb — Amazon Bedrock

Resume el PDF earth.pdf incluido en esta subcarpeta.

Ejecuta las celdas en orden con credenciales AWS y acceso al modelo. La configuración usa servicios reales. [Configuración](../README.md).

In [ ]:
# Cliente real de Amazon Bedrock: credenciales AWS del entorno o perfil.
import os
from pathlib import Path

import boto3
from dotenv import load_dotenv

repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "BEDROCK" / "README.md").is_file()
)
bedrock_root = repo_root / "BEDROCK"
load_dotenv(bedrock_root / ".env", override=False)
exercise_dir = bedrock_root / "FeatsOfClaude"
os.chdir(exercise_dir)

session = boto3.Session(
    profile_name=os.getenv("AWS_PROFILE"),
    region_name=os.getenv("AWS_REGION", "us-west-2"),
)
client = session.client("bedrock-runtime")
model_id = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-6")


In [ ]:
# Mensajes y respuestas en el formato nativo de Converse.
def message_content(content):
    if isinstance(content, str):
        if not content.strip():
            raise ValueError("El texto del mensaje no puede estar vacío.")
        return [{"text": content}]
    if not isinstance(content, list) or not content:
        raise ValueError("El mensaje necesita una lista de bloques de contenido.")
    for block in content:
        if "text" in block and not block["text"].strip():
            raise ValueError("Los bloques de texto no pueden estar vacíos.")
    return content


def add_user_message(messages, content):
    messages.append({"role": "user", "content": message_content(content)})


def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": message_content(content)})


def chat(
    messages,
    system=None,
    stop_sequences=None,
    tools=None,
    tool_choice="auto",
    max_tokens=4000,
):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {"maxTokens": max_tokens},
    }
    if stop_sequences:
        params["inferenceConfig"]["stopSequences"] = list(stop_sequences)
    if system:
        params["system"] = [{"text": system}]
    if tools:
        choices = {"auto": {"auto": {}}, "any": {"any": {}}}
        choice = choices.get(tool_choice, {"tool": {"name": tool_choice}})
        selected_tools = list(tools)
        params["toolConfig"] = {"tools": selected_tools, "toolChoice": choice}
    response = client.converse(**params)
    parts = response["output"]["message"]["content"]
    text = "\n".join(part["text"] for part in parts if "text" in part)
    return {
        "parts": parts,
        "stop_reason": response["stopReason"],
        "text": text,
        "usage": response["usage"],
    }


In [ ]:
pdf_path = exercise_dir / "earth.pdf"
file_bytes = pdf_path.read_bytes()
messages = []
add_user_message(messages, [
    {"document": {"format": "pdf", "name": "earth", "source": {"bytes": file_bytes}}},
    {"text": "Summarize this document in one sentence."},
])
response = chat(messages)
response["text"]
